# Section 6 — CPU float64 gate

Only the registered CPU checks. Exact references and relative comparisons run again. InfSSM remains DEFERRED until phase two.

Same setup flow as H4: Internet ON, Kaggle Secret `github_token`, T4 x2. Source is cloned from GitHub and checked out at a fixed commit. Attach official pretrained weights. Original ImageNet-R images or imagenet-r.tar are optional Inputs: all Input files are searched automatically. If filenames have been changed (as in my1nonly/imagenet-r), the notebook automatically downloads the author archive (about2.19GB) with Internet ON. It preserves the exact Mamba-CL24000/6000 split and200 classes; it never guesses a mapping from renamed files. Previous Output is for resume and for carrying01 selection into02. Checkpoints are compact; large immutable contexts/curvature can remain on earlier Output Inputs. Keep the shards listed in resume_dependencies.json attached. Archives are optional; the default saves the Output tree directly. No source ZIP is required.

Every5 optimizer steps, live output shows throughput, peak VRAM, loss, GPU utilization and exposed dataloader wait. progress.json updates each step; resume still uses the last committed epoch. Heartbeats show /kaggle/working usage and request PAUSED below2GiB free. Before01 launches either lane, an isolated20-step MambaCL probe on GPU0 must reach10 images/s; slower results stop for review. CPU checks pass; actual GPU throughput remains to be measured on Kaggle.

In [ ]:
BRANCH = 'GL'
REPO_COMMIT = '97d4f2f5acba4107b1374cd4837a504a6aa858de'
RUN_TAG = 'glcl_imagenetr_s0_97d4f2f5acba'
STAGE = 'tests'
DATA_ROOT = None  # optional raw OR official train/val root; auto-discovered under /kaggle/input
PRETRAINED = None  # optional explicit path to defocus_mamba_large_cls_21k.pth
JOBS = [[], []]
import time
SESSION_START = time.time()

In [ ]:
STORAGE_PREFLIGHT = '"""Recover generated scratch space without deleting valid experiment checkpoints."""\nimport argparse\nimport json\nimport os\nimport pickletools\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\n\ndef checkpoint_position(path):\n    # Bootstrap is deliberately stdlib-only, before the pinned repo/venv exists.\n    with zipfile.ZipFile(path) as archive:\n        name = next(n for n in archive.namelist() if n.endswith(\'/data.pkl\'))\n        with archive.open(name) as stream:\n            prefix = stream.read(256*1024)\n    fields, pending = {}, None\n    for op, arg, _ in pickletools.genops(prefix):\n        if op.name in (\'BINUNICODE\', \'SHORT_BINUNICODE\', \'UNICODE\') and arg in (\'task\', \'epoch\', \'task_complete\'):\n            pending = arg\n        elif pending and op.name in (\'BININT\', \'BININT1\', \'BININT2\', \'INT\', \'NEWTRUE\', \'NEWFALSE\'):\n            fields[pending] = True if op.name == \'NEWTRUE\' else False if op.name == \'NEWFALSE\' else arg\n            pending = None\n            if len(fields) == 3:\n                return (int(fields[\'task\']), int(fields[\'epoch\']), int(fields[\'task_complete\']))\n    raise ValueError(\'Missing checkpoint position\')\n\n\ndef active_gl_processes():\n    found = []\n    proc = Path(\'/proc\')\n    if proc.is_dir():\n        for path in proc.iterdir():\n            if not path.name.isdigit():\n                continue\n            try:\n                words = (path / \'cmdline\').read_bytes().split(b\'\\0\')\n            except OSError:\n                continue\n            if any(words[i:i+2] in ([b\'-m\', b\'glcl.runner\'], [b\'-m\', b\'glcl.kaggle\']) for i in range(len(words)-1)):\n                found.append(int(path.name))\n    return found\n\n\ndef migrate_directory(source, destination, events):\n    if not source.is_dir() or source.resolve() == destination.resolve():\n        return\n    if destination.exists():\n        # Never overwrite another environment/cache. Retain the old one under\n        # ephemeral storage so the new canonical cache can be validated first.\n        alternate = destination.with_name(destination.name + \'-previous-working\')\n        if alternate.exists():\n            raise RuntimeError(\'Both migration targets exist; inspect storage manually: \' + str(alternate))\n        destination = alternate\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    shutil.move(str(source), str(destination))\n    events.append(dict(moved_from=str(source), moved_to=str(destination)))\n    # Pip is invoked as python -m pip. Repair console-script shebangs as well.\n    if (destination / \'pyvenv.cfg\').is_file():\n        for file in (destination / \'bin\').iterdir():\n            if file.is_file() and not file.is_symlink():\n                data = file.read_bytes()\n                if data.startswith((\'#!\' + str(source)).encode()):\n                    file.write_bytes(data.replace(str(source).encode(), str(destination).encode(), 1))\n\n\ndef prepare(working, ephemeral, repo=None):\n    working, ephemeral = Path(working).absolute(), Path(ephemeral).absolute()\n    running = active_gl_processes()\n    if running:\n        raise RuntimeError(\'A previous GL launcher/worker is still active (PIDs \' + str(running)\n                           + \'). Stop the failed run before repairing its scratch files.\')\n    output = working / \'glcl_outputs\'\n    output.mkdir(parents=True, exist_ok=True)\n    ephemeral.mkdir(parents=True, exist_ok=True)\n    before = shutil.disk_usage(working).free\n    removed, events, recovered = [], [], []\n    paths = [p for p in output.rglob(\'*.tmp\')\n             if p.name.endswith((\'.pt.tmp\', \'.json.tmp\'))]\n    paths += list(output.glob(\'*/*/contexts/context-pending.pt\'))\n    paths.append(working / \'glcl_outputs.tar.tmp\')\n    for path in paths:\n        if path.is_file() or path.is_symlink():\n            if path.name == \'latest.pt.tmp\' and zipfile.is_zipfile(path):\n                try:\n                    rank = checkpoint_position(path)\n                    with zipfile.ZipFile(path) as archive:\n                        if archive.testzip() is not None:\n                            raise ValueError(\'Temporary checkpoint CRC failed\')\n                    current = path.with_suffix(\'\')\n                    try:\n                        old_rank = checkpoint_position(current)\n                    except (OSError, ValueError, zipfile.BadZipFile, StopIteration):\n                        old_rank = (-1, -1, -1)\n                    if rank > old_rank:\n                        os.replace(path, current)\n                        recovered.append(dict(path=str(current), task=rank[0]+1, epoch=rank[1]))\n                        continue\n                except (OSError, ValueError, zipfile.BadZipFile, StopIteration):\n                    pass\n            size = path.stat().st_size if path.exists() else 0\n            path.unlink()\n            removed.append(dict(path=str(path), bytes=size))\n    # The downloaded tar is a reproducible cache, not a checkpoint. Remove it\n    # only if every official image has already been extracted and is present.\n    tools = Path(repo)/\'upstream/mamba-cl/tools\' if repo else None\n    if tools and all((tools/f\'imagenet_r_{mode}.txt\').is_file() for mode in (\'train\', \'val\')):\n        names = [line for mode in (\'train\', \'val\') for line in (tools/f\'imagenet_r_{mode}.txt\').read_text().splitlines()]\n        for base in (working/\'input_data/original_imagenet_r\', ephemeral/\'input_data/original_imagenet_r\'):\n            archive = base/\'imagenet-r.tar\'\n            if archive.is_file() and len(names) == 30000 and all((base/\'images\'/name).is_file() for name in names):\n                size = archive.stat().st_size\n                archive.unlink()\n                removed.append(dict(path=str(archive), bytes=size, reason=\'Verified extracted original images; redundant download cache\'))\n    legacy_ephemeral = working.parent.parent / \'temp/glcl\'\n    if legacy_ephemeral.resolve() != ephemeral.resolve():\n        for name in (\'env\', \'input_data\', \'cache\', \'python\', \'legacy_resume\'):\n            migrate_directory(legacy_ephemeral/name, ephemeral/name, events)\n    migrate_directory(working.parent / \'glcl_env\', ephemeral / \'env\', events)\n    migrate_directory(working / \'input_data\', ephemeral / \'input_data\', events)\n    result = dict(removed_failed_temporary_files=removed, recovered_complete_temporary_checkpoints=recovered, migrations=events,\n                  preserved=\'All latest.pt files, immutable contexts/curvature and valid archives\',\n                  working_free_bytes_before=before, working_free_bytes_after=shutil.disk_usage(working).free,\n                  ephemeral_free_bytes=shutil.disk_usage(ephemeral).free)\n    target = output / \'storage_preflight.json\'\n    target.write_text(json.dumps(result, indent=2) + \'\\n\')\n    print(json.dumps(result, indent=2))\n    return result\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--working\', required=True)\n    p.add_argument(\'--ephemeral\', required=True)\n    p.add_argument(\'--repo\')\n    a = p.parse_args()\n    prepare(a.working, a.ephemeral, a.repo)\n\n\nif __name__ == \'__main__\':\n    main()\n'

import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

os.environ.update(TQDM_DISABLE='1', PYTHONUNBUFFERED='1',
                  PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION='python')
MAIN_URL = 'https://github.com/duyh80456-code/Gramian-Laplace-GL-cho-continual-learning-tr-n-Mamba.git'
WORKING = Path('/kaggle/working')
# A previous execution leaves the kernel inside REPO. Always leave that tree
# before removing it, and also recover if a prior run already deleted the cwd.
WORKING.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING)
REPO = Path('/kaggle/working/gramian-laplace')
RUN_ROOT = Path('/kaggle/working/glcl')
EPHEMERAL = Path('/tmp/glcl')
os.environ.update(GL_EPHEMERAL_ROOT=str(EPHEMERAL), UV_CACHE_DIR=str(EPHEMERAL/'cache/uv'),
                  PIP_CACHE_DIR=str(EPHEMERAL/'cache/pip'), TMPDIR=str(EPHEMERAL/'tmp'),
                  UV_PYTHON_INSTALL_DIR=str(EPHEMERAL/'python'), GL_MIN_FREE_BYTES=str(2*1024**3))
(EPHEMERAL/'tmp').mkdir(parents=True, exist_ok=True)
OUTPUT = RUN_ROOT / 'glcl_outputs'
ROOT_OUT = OUTPUT
subprocess.run([sys.executable, '-c', STORAGE_PREFLIGHT, '--working', str(RUN_ROOT),
                '--ephemeral', str(EPHEMERAL), '--repo', str(REPO)], cwd=WORKING, check=True)
# This scientific protocol must not overwrite or resume old fp32 experiments.
if (OUTPUT/'tune').exists() or (OUTPUT/'main').exists():
    identities=list(OUTPUT.glob('*/*/identity.json'))
    if any(json.loads(p.read_text()).get('config',{}).get('protocol') != 'amp_fp16_resize256_tune10_v2' for p in identities):
        previous=RUN_ROOT/('glcl_outputs_before_amp_'+str(int(time.time())))
        OUTPUT.rename(previous)
        print('Preserved old protocol Output at:',previous)
OUTPUT.mkdir(parents=True,exist_ok=True)

def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)

def private_clone(url, destination, commit):
    token = UserSecretsClient().get_secret('github_token').strip()
    if not token:
        raise RuntimeError('Enable Kaggle Secret github_token for this notebook')
    askpass = Path('/kaggle/working/.glcl_git_askpass.py')
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = dict(os.environ, GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
    try:
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', url, str(destination)], cwd=destination.parent, env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'fetch', 'origin', commit], cwd=destination.parent, env=env, check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], cwd=destination.parent, env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

# Only refresh the source checkout. Outputs and environment live elsewhere.
if REPO.exists():
    shutil.rmtree(REPO)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], cwd=WORKING, text=True).strip()
if CODE_COMMIT != REPO_COMMIT:
    raise RuntimeError('Pinned repository revision mismatch')
os.chdir(REPO)
print('Resolved code commit:', CODE_COMMIT, 'pin:', REPO_COMMIT)
print('Official MambaCL:', json.loads((REPO/'upstream/LOCK.json').read_text()))

In [ ]:
# Isolated Python 3.11 + official torch 2.2.2; no Kaggle host torch replacement.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.5.31'], check=True)
VENV = EPHEMERAL/'env'
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                '-r', str(REPO/'requirements-kaggle.txt'), 'pip==24.2'], check=True)

ENV = dict(os.environ, PYTHONPATH=str(REPO), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1', TQDM_DISABLE='1')

In [ ]:
# Recompute the supervisor-corrected section-six checks, same fixture/noises:
# C .852433 > proxy .725509 and >=.8; Delta .968065 > proxy .950543 and >=.85.
# Test5 remains DEFERRED until phase two.
# A failure intentionally stops before model initialization/training.
subprocess.run([PYTHON, '-m', 'tests.section6'], env=ENV, check=True)
print((REPO/'reports/section6.json').read_text())